# Feature Checks

A first-order screen for leakage and low-quality features, and the point where the exclude list is built. The screen **flags, it never drops**. There is **no automated feature selection** — the final feature set is the curated include list minus the exclusions decided here, which get pasted back into the product config.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'configs' / '_schema.py').exists())
sys.path.insert(0, str(ROOT))
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from configs._schema import load_config
cfg = load_config(ROOT / 'configs/fx_activation.yaml')   # validates on load
cfg.product, str(cfg.obs_date), len(cfg.features)

from src.dataset import load_labelled
from src.leakage import leakage_report

X, y = load_labelled(spark, cfg, cfg.obs_date)

## 1. FX Activation

### First-order screen
Per feature: direction-agnostic univariate AUC, % missing, dominance of the most common value, and a `suspected_leak` flag (univariate AUC > 0.95).

In [ ]:
rep = leakage_report(X, y, cfg.features)
rep.head(20)

### Read the flags
High univariate AUC → **investigate, don't reflexively drop**: true signal or a post-outcome proxy? High dominance / high missingness → weak or unstable, usually a drop. Always ask what a feature could plausibly know as-of the observation month.

In [ ]:
rep[rep['suspected_leak']]                     # strong separators to interrogate

In [ ]:
rep[(rep['value_dominance'] > 0.98) | (rep['pct_missing'] > 0.5)]   # usually drop

### Human decision → exclude list
Edit this list by hand after reviewing the tables above, then paste the printed block into `features_exclude` in the config.

In [ ]:
candidate_excludes = [
    'fx_fwd_txn_3m',   # the raw forward signal the target was built from — a leak
    # add features you judged to leak or to be worthless
]
print('features_exclude:')
for f in candidate_excludes:
    print(f'  - {f}')

### Redundancy — optional, cheap
Correlation among the survivors; aids interpretability and card stability, not correctness. Not a selection step.

In [ ]:
import numpy as np
surv = [f for f in cfg.features if f not in candidate_excludes]
corr = X[surv].select_dtypes('number').corr().abs()
pairs = (corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
         .stack().sort_values(ascending=False))
pairs[pairs > 0.9].head(15)

**Reminder:** no automated in-CV selection in the crunch. Curated `include − exclude` *is* the feature set. If selection is ever added, it must run inside CV folds — FLAML won't do that for you.